# R-CNN: Rich Feature Hierarchies for Accurate Object Detection

**Paper**: Girshick et al., CVPR 2014

## The Two-Stage Detection Family

<img src='../figures/rcnn_family_summary.png' width='760'/>

R-CNN, Fast R-CNN, and Faster R-CNN form the canonical two-stage detector family. Each solved the key bottleneck of the previous:
- **R-CNN (2014)**: CNN features beat HOG, but 47 sec/image — too slow
- **Fast R-CNN (2015)**: Shared CNN feature map → 2.3 sec/image
- **Faster R-CNN (2016)**: Replace Selective Search with learned RPN → 0.2 sec/image

## R-CNN Pipeline

<img src='../figures/rcnn_pipeline.png' width='700'/>

**Stage 1 — Region Proposals**: Selective Search generates ~2000 candidate regions per image by hierarchically merging superpixels based on color, texture, and size.

**Stage 2 — Feature Extraction**: Each proposal is warped to 227x227 and forwarded through AlexNet/VGG16 independently → 4096-dim feature vector.

**Stage 3 — Classify + Regress**:
- One linear SVM per class scores each region
- One bounding-box regressor per class refines box coordinates
- NMS removes duplicate detections per class

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import requests, torch.nn as nn
from io import BytesIO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Selective Search Simulation

Real Selective Search uses `cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()`. We simulate it by sampling multi-scale boxes — showing what ~2000 proposals cover.

In [ ]:
def simulate_selective_search(img_w, img_h, n=80):
    np.random.seed(42)
    boxes = []
    for _ in range(n):
        scale = np.random.choice([0.1,0.2,0.3,0.5,0.7])
        w = min(int(img_w*scale*np.random.uniform(0.5,2.0)), img_w-1)
        h = min(int(img_h*scale*np.random.uniform(0.5,2.0)), img_h-1)
        x = np.random.randint(0, max(1, img_w-w))
        y = np.random.randint(0, max(1, img_h-h))
        boxes.append([x,y,x+w,y+h])
    return boxes

def warp_region(img, box, size=227):
    x1,y1,x2,y2 = [int(v) for v in box]
    return img.crop((x1,y1,max(x2,x1+1),max(y2,y1+1))).resize((size,size),Image.BILINEAR)

try:
    img = Image.open(BytesIO(requests.get('https://ultralytics.com/images/zidane.jpg',timeout=8).content)).convert('RGB')
except Exception:
    img = Image.fromarray(np.random.randint(0,255,(480,640,3),dtype=np.uint8))
print(f'Image: {img.size}')

proposals = simulate_selective_search(*img.size, n=50)
warped    = [warp_region(img, b) for b in proposals[:6]]

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].imshow(img)
for b in proposals[:30]:
    x1,y1,x2,y2=b
    axes[0].add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1,edgecolor='yellow',facecolor='none',alpha=0.5))
axes[0].set_title('Region Proposals (simulated Selective Search)'); axes[0].axis('off')
axes[1].imshow(np.hstack([np.array(w) for w in warped]))
axes[1].set_title('Each proposal warped to 227x227 → separate CNN forward pass'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print('2000 proposals = 2000 forward passes per image (~47 sec on GPU!)')

## VGG16 as Feature Extractor

In [ ]:
class RCNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = torchvision.models.vgg16(weights='DEFAULT')
        self.features = vgg.features
        self.avgpool  = vgg.avgpool
        self.fc       = nn.Sequential(*list(vgg.classifier.children())[:4])

    def forward(self, x):
        return self.fc(torch.flatten(self.avgpool(self.features(x)),1))

extractor = RCNNFeatureExtractor().to(device).eval()
tf = T.Compose([T.Resize((224,224)), T.ToTensor(),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
batch = torch.stack([tf(warp_region(img,b)) for b in proposals[:5]]).to(device)
with torch.no_grad():
    feats = extractor(batch)
print(f'Input:   {tuple(batch.shape)}')
print(f'Features:{tuple(feats.shape)}  → fed to per-class SVM + bbox regressor')

## Bounding Box Regression + NMS

In [ ]:
def encode_boxes(p,g):
    px,py=(p[:,0]+p[:,2])/2,(p[:,1]+p[:,3])/2
    pw,ph=p[:,2]-p[:,0],p[:,3]-p[:,1]
    gx,gy=(g[:,0]+g[:,2])/2,(g[:,1]+g[:,3])/2
    gw,gh=g[:,2]-g[:,0],g[:,3]-g[:,1]
    return np.stack([(gx-px)/pw,(gy-py)/ph,np.log(gw/pw),np.log(gh/ph)],1)

def decode_boxes(p,d):
    px,py=(p[:,0]+p[:,2])/2,(p[:,1]+p[:,3])/2
    pw,ph=p[:,2]-p[:,0],p[:,3]-p[:,1]
    rx,ry=d[:,0]*pw+px,d[:,1]*ph+py
    rw,rh=np.exp(d[:,2])*pw,np.exp(d[:,3])*ph
    return np.stack([rx-rw/2,ry-rh/2,rx+rw/2,ry+rh/2],1)

def iou(a,b):
    ix1,iy1=max(a[0],b[0]),max(a[1],b[1])
    ix2,iy2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    return inter/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter+1e-6)

def nms(boxes,scores,thresh=0.5):
    order=np.argsort(scores)[::-1]; keep=[]
    while len(order):
        i=order[0]; keep.append(i)
        ious=np.array([iou(boxes[i],boxes[j]) for j in order[1:]])
        order=order[1:][ious<thresh]
    return keep

# Bbox regression demo
p=np.array([[100.,100.,200.,200.]]); g=np.array([[110.,95.,210.,210.]])
d=encode_boxes(p,g)
print(f'Proposal:  {p[0]}'); print(f'GT:        {g[0]}')
print(f'Delta:     {d[0].round(4)}')
print(f'Decoded:   {decode_boxes(p,d)[0].round(1)}  (matches GT)')

# NMS demo
boxes=np.array([[50,50,150,150],[60,55,155,155],[65,60,160,160],[300,300,400,400]],dtype=float)
scores=np.array([0.9,0.75,0.6,0.85])
kept=nms(boxes,scores)
fig,axes=plt.subplots(1,2,figsize=(10,4))
colors=['red','orange','gold','royalblue']
for ax,(title,idxs) in zip(axes,[('Before NMS',range(4)),('After NMS',kept)]):
    canvas=np.ones((500,500,3),dtype=np.uint8)*230
    ax.imshow(canvas)
    for i in idxs:
        x1,y1,x2,y2=boxes[i]
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2.5,edgecolor=colors[i],facecolor='none'))
        ax.text(x1,y1-6,f'{scores[i]:.2f}',color=colors[i],fontsize=11,fontweight='bold')
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## Summary

| Model | Speed | mAP VOC07 | Key bottleneck fixed |
|-------|-------|-----------|---------------------|
| **R-CNN** | ~47 sec/img | 66.0% | — (baseline) |
| **Fast R-CNN** | ~2.3 sec/img | 70.0% | Shared feature map via RoI Pooling |
| **Faster R-CNN** | ~0.2 sec/img | 73.2% | Learned proposals (RPN) |

R-CNN established the pipeline. Every improvement (Fast R-CNN, Faster R-CNN, FPN, Mask R-CNN) is an evolution of this same framework.